In [1]:
import os
import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {"sample_title": sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':')
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


# Create metadata DataFrame
metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'
metadata_df['sample_title'] = metadata_df['sample_title'].str.replace('.RNA-seq', '', regex=False)
metadata_df.head()

,sample_title,diagnosis,tissue,Sex,age,Stage,cell type,differentiation,grade,survstatus,time to last followup
samples,,,,,,,,,,,
GSM2328306,Lu1031A,Lung Adenocarcinoma,Lung,Female,57,IA,AD,Well,II,0,8.59
GSM2328307,Lu1031B,Lung Adenocarcinoma,Lung,Female,57,IA,NL,Well,II,0,8.59
GSM2328308,Lu1050A,Lung Adenocarcinoma,Lung,Male,58,IB,AD,Well,II,0,7.54
GSM2328309,Lu1050B,Lung Adenocarcinoma,Lung,Male,58,IB,NL,Well,II,0,7.54
GSM2328310,Lu1068A,Lung Adenocarcinoma,Lung,Female,57,IB,AD,Mod,III,1,3.28


# Use NCBI provided expression data

In [2]:
# Raw counts
ncbi_data_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc={GEO_ID}&format=file&file={GEO_ID}_raw_counts_GRCh38.p13_NCBI.tsv.gz"

df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None

url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["GeneID", "Symbol"]]
    .dropna(subset=["GeneID", "Symbol"])
    .set_index("GeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in df_raw_counts.columns if c in gene_mapping]
df_raw_counts = df_raw_counts[mappable].rename(columns=gene_mapping)
# df_raw_counts = df_raw_counts.rename(columns=gene_mapping)


data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'

df_combined = metadata_df.join(df_raw_counts, how='inner')

# Drop rows where "cell type" is 'NL' (normal lung)
df_combined = df_combined[df_combined["cell type"] != "NL"].copy()


df_combined.reset_index()
df_combined.to_csv(data_file_name, index=False)

df_combined

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_11666/1592177144.py:10: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


,sample_title,diagnosis,tissue,Sex,age,Stage,cell type,differentiation,grade,survstatus,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
samples,,,,,,,,,,,,,,,,,,,,,
GSM2328306,Lu1031A,Lung Adenocarcinoma,Lung,Female,57,IA,AD,Well,II,0,...,365260,557,385,519,203685,101559,9163,210836,1026,2160
GSM2328308,Lu1050A,Lung Adenocarcinoma,Lung,Male,58,IB,AD,Well,II,0,...,1056970,704,488,655,437215,190437,10637,696762,382,2225
GSM2328310,Lu1068A,Lung Adenocarcinoma,Lung,Female,57,IB,AD,Mod,III,1,...,874756,450,398,458,649144,290787,18293,468782,736,3242
GSM2328312,Lu106A,Lung Adenocarcinoma,Lung,Male,63,IB,AD,Well,II,0,...,256091,280,212,263,82629,45347,3832,192436,185,1665
GSM2328314,Lu113A,Lung Adenocarcinoma,Lung,Female,89,IA,AD,Well,II,1,...,314422,533,280,436,193412,84158,6827,200220,846,2973
GSM2328316,Lu1182A,Lung Adenocarcinoma,Lung,Female,61,IA,BAC,Well,II,0,...,743922,540,350,497,483866,263197,18422,498737,574,4436
GSM2328318,Lu1321A,Lung Adenocarcinoma,Lung,Female,68,IB,AD,Well,II,0,...,721498,362,278,340,382739,216379,15148,394502,552,2470
GSM2328320,Lu1377A,Lung Adenocarcinoma,Lung,Female,64,IA,AD,Well,II,0,...,423379,192,149,187,157171,62310,3958,174388,143,1433
GSM2328322,Lu1405A,Lung Adenocarcinoma,Lung,Female,54,IB,AD,Mod,III,0,...,354124,260,195,217,214850,91765,5429,181855,190,2020


# Generate description

In [3]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_geo

description_file_path = 'description.json'


if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_geo(GEO_ID)

desc_data['data_summary'] = {
    'samples': metadata_df.shape[0],
    'clinical_features': metadata_df.shape[1],
    'genes': df_raw_counts.shape[1]
}
print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)

Description updated


# Use author provided expression data


In [4]:
metadata_df = metadata_df.reset_index()
metadata_df = metadata_df.set_index('sample_title')

# download this expression matricies
raw_exp_url = 'https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE87340&format=file&file=GSE87340%5FRawExpr%2Etxt%2Egz'
processed_exp_url = 'https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE87340&format=file&file=GSE87340%5FRPKM%5Flog2%2Etxt%2Egz'


df_raw_exp: pd.DataFrame = pd.read_csv(raw_exp_url, sep='\t', index_col=0, compression='gzip')
df_raw_exp = df_raw_exp[df_raw_exp.index.notna()]
df_raw_exp= df_raw_exp.T
df_raw_exp.index.name = "samples"

df_processed_exp: pd.DataFrame = pd.read_csv(processed_exp_url, sep='\t', index_col=0, compression='gzip')
df_processed_exp = df_processed_exp[df_processed_exp.index.notna()]
df_processed_exp = df_processed_exp.T
df_processed_exp.index.name = "samples"

df_combined_raw = metadata_df.join(df_raw_exp, how='inner')
df_combined_raw.reset_index(inplace=True)
# INFO: Uncomment to save the data to disk
# df_combined_raw.to_csv(f'{GEO_ID}_raw.csv', index=False)

df_combined_processed = metadata_df.join(df_processed_exp, how='inner')
df_combined_processed.reset_index(inplace=True)
# INFO: Uncomment to save the data to disk
# df_combined_processed.to_csv(f'{GEO_ID}_processed.csv', index=False)

df_combined_raw.head()

,sample_title,samples,diagnosis,tissue,Sex,age,Stage,cell type,differentiation,grade,...,MOGAT2,TAS2R50,BLOC1S1,C1QTNF9B,TRAPPC5,LOC100329109,MPV17,LOC100507254,GFRA1,USP9Y
0,Lu1031A,GSM2328306,Lung Adenocarcinoma,Lung,Female,57,IA,AD,Well,II,...,0.0,2.0,5198.0,4.0,4286.0,676.0,4286.0,32.0,2942.0,16.0
1,Lu1031B,GSM2328307,Lung Adenocarcinoma,Lung,Female,57,IA,NL,Well,II,...,0.0,0.0,5292.0,18.0,3178.0,950.0,3590.0,78.0,7172.0,8.0
2,Lu1050A,GSM2328308,Lung Adenocarcinoma,Lung,Male,58,IB,AD,Well,II,...,6.0,4.0,7862.0,14.0,2156.0,802.0,5308.0,6.0,6006.0,4520.0
3,Lu1050B,GSM2328309,Lung Adenocarcinoma,Lung,Male,58,IB,NL,Well,II,...,2.0,2.0,4628.0,10.0,2854.0,938.0,3688.0,14.0,3630.0,5316.0
4,Lu1068A,GSM2328310,Lung Adenocarcinoma,Lung,Female,57,IB,AD,Mod,III,...,6.0,4.0,4516.0,28.0,3048.0,732.0,6284.0,12.0,3770.0,20.0


In [5]:
# test = metadata_df.join(df_processed_exp.loc[:, df_processed_exp.quantile(0.75) > 0], how='inner')
# test.reset_index(inplace=True)
# test.to_csv(f'{GEO_ID}_processed_filtered.csv', index=False)